# TCN LOSO Baseline: Binary PREICTAL vs INTERICTAL (temporal history model)

**Task scope.** Third modeling experiment on the patient-specific LOSO
pipeline, after Logistic Regression (`EEG_logistic_loso_binary.ipynb`) and
Random Forest (`EEG_random_forest_loso_binary.ipynb`). Tests whether
**explicit temporal history** (a causal, dilated TCN over consecutive
30-second epochs) improves seizure forecasting over the two static,
single-epoch baselines. **Not** a redesign of labels/horizons/LOSO/
normalization -- only the estimator and its input shape (a sequence instead
of a single epoch) change.

**This is a NEW notebook, independent of the historical 4-class TCN
notebooks** (`EEG_TCN_seizure_forecasting.ipynb`,
`EEG_TCN_state_deviation_forecasting.ipynb`), which remain untouched,
historical-only, and are not read or modified here. This notebook's
architecture style (causal dilated Conv1d blocks) is a fresh
reimplementation inspired by that history, not an import of its code --
credited explicitly in Section 6 below, and deliberately NOT reusing its
4-class target, its pooled-scaler normalization, or its recording-level
TRAIN/VAL/TEST split (all superseded by the binary/LOSO/patient-specific
methodology already validated in Sections 18-22 of project_state.md).

**Reused, unmodified inputs (verbatim, credited, copied from
`EEG_logistic_loso_binary.ipynb` / `EEG_random_forest_loso_binary.ipynb`):**
- `outputs/labels_binary_preictal.parquet` -- RAW (unscaled) qEEG features +
  binary labels for all three horizons.
- Fold-construction functions (`compute_seizure_eligibility`,
  `assign_fold_context`, `fit_patient_scalers`,
  `transform_with_patient_scalers`, `build_loso_fold`, `validate_loso_fold`).
- Metric functions (`fold_seizure_event_metrics`, `fold_epoch_metrics`,
  `pool_epoch_metrics`, `pool_event_metrics`).

**New in this notebook:** sequence construction from consecutive epochs
under a strict same-split, same-recording, no-future-information rule
(Section 4-5); a compact causal TCN (Section 6); a TRAIN-only validation
split for early stopping (Section 7); a per-fold training loop (Section 8);
and a three-way LR vs RF vs TCN comparison (Section 16-17) built by reading
all three models' own saved outputs.

**New output:** everything under `outputs/modeling/tcn_loso_binary/`.
Nothing existing is modified.


## 1. Setup and cross-check against LR/RF configuration


In [1]:

from pathlib import Path
import json as _json

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, precision_recall_curve

print("=" * 70)
print("TCN LOSO BINARY BASELINE -- SETUP")
print("=" * 70)
print(f"numpy : {np.__version__}")
print(f"pandas: {pd.__version__}")
import sklearn
print(f"sklearn: {sklearn.__version__}")
import torch
print(f"torch: {torch.__version__} (CUDA available: {torch.cuda.is_available()})")

repo_root = Path.cwd()
output_dir = repo_root / "outputs"
run_dir = output_dir / "modeling" / "tcn_loso_binary"
plot_dir = run_dir / "plots"
run_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

lr_run_dir = output_dir / "modeling" / "logistic_loso_binary"
rf_run_dir = output_dir / "modeling" / "random_forest_loso_binary"
for d in (lr_run_dir, rf_run_dir):
    if not d.exists():
        raise FileNotFoundError(f"{d} not found -- run the LR and RF notebooks first (reference baselines).")

in_path = output_dir / "labels_binary_preictal.parquet"
if not in_path.exists():
    raise FileNotFoundError(f"{in_path} not found -- run EEG_binary_preictal_labeling.ipynb first.")
df = pd.read_parquet(in_path)
print(f"\nLoaded {in_path.name}: shape={df.shape}")

PATIENT_COL = "patient_id"
HORIZONS = [5, 15, 30]
NON_FEATURE_COLS = {
    "patient_id", "subject_id", "recording_id", "dataset", "epoch_idx",
    "t_start_sec", "t_center_sec", "t_end_sec",
    "y_seizure", "seizure_id", "next_seizure_id", "next_seizure_start_sec",
    "time_to_next_seizure_sec", "previous_seizure_id",
    "previous_seizure_end_sec", "time_since_previous_seizure_sec",
    "binary_label_h5", "binary_label_h15", "binary_label_h30",
    "exclusion_reason_h5", "exclusion_reason_h15", "exclusion_reason_h30",
}
FEATURE_COLS = [c for c in df.columns if c not in NON_FEATURE_COLS]
PATIENTS = sorted(df[PATIENT_COL].unique())
print(f"\nFeature columns used ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Patients (modeled SEPARATELY, never pooled): {PATIENTS}")

EPOCH_SEC = float((df["t_end_sec"] - df["t_start_sec"]).round(3).mode().iloc[0])
assert abs(EPOCH_SEC - 30.0) < 1e-6, f"Expected 30s epochs (project_state.md 5.1), got {EPOCH_SEC}"
print(f"Epoch duration (from data): {EPOCH_SEC}s")

RANDOM_STATE = 42
THRESHOLD = 0.5
print(f"\nRANDOM_STATE={RANDOM_STATE}, decision THRESHOLD={THRESHOLD} (fixed, not tuned on TEST)")

with open(lr_run_dir / "experiment_config.json") as _f:
    _lr_config = _json.load(_f)
with open(rf_run_dir / "experiment_config.json") as _f:
    _rf_config = _json.load(_f)
assert FEATURE_COLS == _lr_config["feature_cols"] == _rf_config["feature_cols"]
assert HORIZONS == _lr_config["horizons_minutes"] == _rf_config["horizons_minutes"]
assert PATIENTS == _lr_config["patients"] == _rf_config["patients"]
print("\n[OK] FEATURE_COLS, HORIZONS, and PATIENTS are byte-for-byte identical to both the LR and RF "
      "baselines' experiment_config.json files.")


TCN LOSO BINARY BASELINE -- SETUP
numpy : 2.4.6
pandas: 3.0.5
sklearn: 1.9.1


torch: 2.14.0+cpu (CUDA available: False)

Loaded labels_binary_preictal.parquet: shape=(2040, 36)

Feature columns used (14): ['std', 'var', 'rms', 'delta', 'theta', 'alpha', 'beta', 'rel_delta', 'rel_theta', 'rel_alpha', 'rel_beta', 'theta_alpha_ratio', 'delta_beta_ratio', 'entropy']
Patients (modeled SEPARATELY, never pooled): ['chb12', 'chb24']
Epoch duration (from data): 30.0s

RANDOM_STATE=42, decision THRESHOLD=0.5 (fixed, not tuned on TEST)

[OK] FEATURE_COLS, HORIZONS, and PATIENTS are byte-for-byte identical to both the LR and RF baselines' experiment_config.json files.


## 2. Fold-construction functions

Copied verbatim from `EEG_logistic_loso_binary.ipynb` /
`EEG_random_forest_loso_binary.ipynb`. Byte-for-byte the same LOSO
construction already leakage-validated (57/57 folds, 9 checks each). This
notebook's TCN folds start from the exact same `train_idx`/`test_idx` as the
LR/RF folds -- sequence construction (Section 5) then *restricts* those
index sets further (to same-split, same-recording, contiguous windows), it
never redefines them.


In [2]:

def compute_seizure_eligibility(df, patient_id, horizon_minutes, seizure_ids):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)'''
    lab_col = f"binary_label_h{horizon_minutes}"
    reason_col = f"exclusion_reason_h{horizon_minutes}"
    horizon_sec = horizon_minutes * 60.0
    pdf = df[df["patient_id"] == patient_id]

    eligible, ineligible = [], []
    for sid in seizure_ids:
        n_preictal = int(((pdf[lab_col] == "PREICTAL") & (pdf["next_seizure_id"] == sid)).sum())
        if n_preictal > 0:
            eligible.append(sid)
            continue
        candidates = pdf[
            (pdf["next_seizure_id"] == sid)
            & (pdf["time_to_next_seizure_sec"] >= 0)
            & (pdf["time_to_next_seizure_sec"] <= horizon_sec)
        ]
        if len(candidates) == 0:
            reason = "no candidate epochs within horizon (insufficient recording history before onset)"
        else:
            reason_counts = candidates[reason_col].value_counts()
            top_reason = reason_counts.idxmax() if len(reason_counts) else "unknown"
            reason = f"all {len(candidates)} candidate epoch(s) excluded (dominant reason: {top_reason})"
        ineligible.append({"seizure_id": sid, "reason": reason})
    return eligible, ineligible


def assign_fold_context(df, horizon_minutes):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)'''
    lab_col = f"binary_label_h{horizon_minutes}"
    fold_context = pd.Series(None, index=df.index, dtype=object)

    is_preictal = (df[lab_col] == "PREICTAL").to_numpy()
    is_interictal = (df[lab_col] == "INTERICTAL").to_numpy()

    assert df.loc[is_preictal, "next_seizure_id"].notna().all(), \
        "A PREICTAL epoch has no next_seizure_id -- cannot assign fold context"
    fold_context.loc[is_preictal] = df.loc[is_preictal, "next_seizure_id"]

    inter_next = df.loc[is_interictal, "next_seizure_id"]
    inter_prev = df.loc[is_interictal, "previous_seizure_id"]
    inter_context = inter_next.where(inter_next.notna(), inter_prev)
    assert inter_context.notna().all(), \
        "An INTERICTAL epoch has neither a next nor a previous seizure in its recording"
    fold_context.loc[is_interictal] = inter_context

    return fold_context


def fit_patient_scalers(df, patient_col, feature_cols, train_mask):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_logistic_loso_binary.ipynb.)'''
    scalers = {}
    train_mask = np.asarray(train_mask)
    for pid in df.loc[train_mask, patient_col].unique():
        sel = train_mask & (df[patient_col] == pid).to_numpy()
        n = int(sel.sum())
        if n == 0:
            raise ValueError(f"Patient {pid!r} has zero TRAIN rows -- cannot fit a scaler.")
        assert df.loc[sel, patient_col].nunique() == 1
        scaler = StandardScaler()
        scaler.fit(df.loc[sel, feature_cols])
        scalers[pid] = scaler
    return scalers


def transform_with_patient_scalers(df, patient_col, feature_cols, scalers):
    '''(Verbatim from EEG_patient_normalization.ipynb, via EEG_logistic_loso_binary.ipynb.)'''
    out = df.copy()
    scaled = np.full((len(df), len(feature_cols)), np.nan)
    covered = np.zeros(len(df), dtype=bool)
    for pid, scaler in scalers.items():
        sel = (df[patient_col] == pid).to_numpy()
        if sel.any():
            scaled[sel, :] = scaler.transform(df.loc[sel, feature_cols])
            covered |= sel
    if not covered.all():
        missing_pids = sorted(df.loc[~covered, patient_col].unique())
        raise ValueError(f"No fitted scaler available for patients: {missing_pids}")
    scaled_cols = [f"{c}_scaled" for c in feature_cols]
    out[scaled_cols] = scaled
    return out, scaled_cols


def build_loso_fold(patient_df, horizon_minutes, held_out_seizure_id, fold_context, feature_cols, patient_col="patient_id"):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.)'''
    lab_col = f"binary_label_h{horizon_minutes}"
    labeled_mask = patient_df[lab_col].notna().to_numpy()
    ctx = fold_context.loc[patient_df.index].to_numpy()

    test_mask = labeled_mask & (ctx == held_out_seizure_id)
    train_mask = labeled_mask & (ctx != held_out_seizure_id)

    scalers = fit_patient_scalers(patient_df, patient_col, feature_cols, train_mask)
    scaled_df, scaled_cols = transform_with_patient_scalers(patient_df, patient_col, feature_cols, scalers)

    pid = patient_df[patient_col].iloc[0]
    return {
        "patient_id": pid,
        "horizon_minutes": horizon_minutes,
        "held_out_seizure_id": held_out_seizure_id,
        "lab_col": lab_col,
        "train_idx": patient_df.index[train_mask],
        "test_idx": patient_df.index[test_mask],
        "scaler": scalers[pid],
        "scaled_df": scaled_df,
        "scaled_cols": scaled_cols,
    }


def validate_loso_fold(fold, patient_df, feature_cols):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_loso_evaluation_splitting.ipynb.) 9 leakage checks.'''
    train_idx, test_idx = fold["train_idx"], fold["test_idx"]
    lab_col = fold["lab_col"]
    held_out = fold["held_out_seizure_id"]

    assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST indices overlap"

    train_next = patient_df.loc[train_idx, "next_seizure_id"]
    train_preictal = patient_df.loc[train_idx, lab_col] == "PREICTAL"
    assert not (train_preictal & (train_next == held_out)).any(), \
        f"Held-out seizure {held_out} has a PREICTAL epoch leaking into TRAIN"

    scaler = fold["scaler"]
    assert int(scaler.n_samples_seen_) == len(train_idx), \
        f"scaler.n_samples_seen_={scaler.n_samples_seen_} != len(train_idx)={len(train_idx)}"
    manual_mean = patient_df.loc[train_idx, feature_cols].to_numpy().mean(axis=0)
    assert np.allclose(manual_mean, scaler.mean_), "scaler mean does not match TRAIN-only recomputation"

    combined = pd.concat([patient_df.loc[train_idx, "patient_id"], patient_df.loc[test_idx, "patient_id"]])
    assert combined.nunique() == 1, "More than one patient present in a single LOSO fold"

    assert list(scaler.feature_names_in_) == list(feature_cols), "Feature column order mismatch at fit time"

    assert patient_df.loc[train_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()
    assert patient_df.loc[test_idx, lab_col].isin(["PREICTAL", "INTERICTAL"]).all()

    assert (patient_df.loc[train_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TRAIN"
    assert (patient_df.loc[test_idx, "y_seizure"] == 0).all(), "A seizure-overlapping epoch is in TEST"

    all_idx = train_idx.union(test_idx)
    for col in ["recording_id", "epoch_idx", "t_start_sec", "t_end_sec"]:
        assert patient_df.loc[all_idx, col].notna().all(), f"Metadata column {col} has nulls in this fold"

    return True


print("[OK] Fold-construction functions reused verbatim from EEG_logistic_loso_binary.ipynb.")


[OK] Fold-construction functions reused verbatim from EEG_logistic_loso_binary.ipynb.


## 3. Metric functions

Copied verbatim from `EEG_logistic_loso_binary.ipynb` /
`EEG_seizure_prediction_metrics.ipynb`. Identical definitions and NaN
behavior to LR and RF -- required for a fair three-way comparison.


In [3]:

REQUIRED_PRED_COLS = [
    "patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id",
    "recording_id", "epoch_idx", "t_start_sec", "t_end_sec",
    "time_to_next_seizure_sec", "y_true", "y_prob", "y_pred",
]

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, average_precision_score,
)


def validate_predictions_df(df, require_single_fold=False):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    missing = [c for c in REQUIRED_PRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"predictions_df missing required columns: {missing}")
    if df["y_true"].isna().any() or not set(df["y_true"].unique()) <= {0, 1}:
        raise ValueError("y_true must be exactly {0, 1} with no NaNs")
    if df["y_pred"].isna().any() or not set(df["y_pred"].unique()) <= {0, 1}:
        raise ValueError("y_pred must be exactly {0, 1} with no NaNs")
    if df["y_prob"].isna().any() or (df["y_prob"] < 0).any() or (df["y_prob"] > 1).any():
        raise ValueError("y_prob must be in [0, 1] with no NaNs")
    if require_single_fold:
        key_cols = ["patient_id", "horizon_minutes", "held_out_seizure_id"]
        if df[key_cols].drop_duplicates().shape[0] != 1:
            raise ValueError("fold_df must contain exactly one (patient, horizon, held-out seizure)")
    return True


def _count_alarm_events(sorted_epoch_idx, sorted_y_pred):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    n = len(sorted_epoch_idx)
    if n == 0:
        return 0
    epoch_idx = np.asarray(sorted_epoch_idx)
    y_pred = np.asarray(sorted_y_pred)
    new_block = np.concatenate(([True], np.diff(epoch_idx) != 1))
    block_id = np.cumsum(new_block)
    n_events = 0
    for b in np.unique(block_id):
        p = y_pred[block_id == b]
        new_run = np.concatenate(([True], np.diff(p) != 0))
        run_id = np.cumsum(new_run)
        for r in np.unique(run_id):
            if p[run_id == r][0] == 1:
                n_events += 1
    return n_events


def fold_seizure_event_metrics(fold_df, epoch_sec=EPOCH_SEC):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    validate_predictions_df(fold_df, require_single_fold=True)

    pre = fold_df[fold_df["y_true"] == 1].sort_values("t_start_sec")
    inter = fold_df[fold_df["y_true"] == 0].sort_values(["recording_id", "epoch_idx"])

    n_preictal_alarms = int((pre["y_pred"] == 1).sum())
    detected = int(n_preictal_alarms > 0)

    if detected:
        alarm_epochs = pre[pre["y_pred"] == 1]
        first_alarm = alarm_epochs.loc[alarm_epochs["time_to_next_seizure_sec"].idxmax()]
        time_in_warning_sec = float(first_alarm["time_to_next_seizure_sec"])
    else:
        time_in_warning_sec = np.nan

    n_false_alarm_events = _count_alarm_events(inter["epoch_idx"].to_numpy(), inter["y_pred"].to_numpy())
    n_interictal_epochs = len(inter)
    interictal_hours = n_interictal_epochs * epoch_sec / 3600.0
    false_alarms_per_hour = (
        n_false_alarm_events / interictal_hours if interictal_hours > 0 else np.nan
    )

    return {
        "n_test_preictal": len(pre),
        "n_test_interictal": n_interictal_epochs,
        "seizure_detected": detected,
        "n_preictal_alarm_epochs": n_preictal_alarms,
        "time_in_warning_sec": time_in_warning_sec,
        "n_false_alarm_events": n_false_alarm_events,
        "interictal_hours": interictal_hours,
        "false_alarms_per_hour": false_alarms_per_hour,
    }


def fold_epoch_metrics(fold_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    validate_predictions_df(fold_df, require_single_fold=False)

    y_true = fold_df["y_true"].to_numpy()
    y_pred = fold_df["y_pred"].to_numpy()
    y_prob = fold_df["y_prob"].to_numpy()

    n_classes_true = len(np.unique(y_true))

    accuracy = float(accuracy_score(y_true, y_pred))
    precision = float(precision_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    recall = float(recall_score(y_true, y_pred, pos_label=1, zero_division=np.nan))
    f1 = float(f1_score(y_true, y_pred, pos_label=1, zero_division=np.nan))

    if n_classes_true < 2:
        auc_pr = np.nan
        auc_pr_undefined_reason = "only one class present in y_true (AUC-PR is not meaningful)"
    else:
        auc_pr = float(average_precision_score(y_true, y_prob))
        auc_pr_undefined_reason = None

    return {
        "n_epochs": len(fold_df),
        "n_classes_in_test": n_classes_true,
        "accuracy": accuracy,
        "precision_preictal": precision,
        "recall_preictal": recall,
        "f1_preictal": f1,
        "auc_pr": auc_pr,
        "auc_pr_undefined_reason": auc_pr_undefined_reason,
    }


def pool_epoch_metrics(pooled_epochs_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    return fold_epoch_metrics(pooled_epochs_df)


def pool_event_metrics(fold_event_metrics_df):
    '''(Verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.)'''
    n_folds = len(fold_event_metrics_df)
    n_detected = int(fold_event_metrics_df["seizure_detected"].sum())
    seizure_sensitivity = n_detected / n_folds if n_folds > 0 else np.nan

    total_fa_events = int(fold_event_metrics_df["n_false_alarm_events"].sum())
    total_interictal_hours = float(fold_event_metrics_df["interictal_hours"].sum())
    pooled_fa_per_hour = (
        total_fa_events / total_interictal_hours if total_interictal_hours > 0 else np.nan
    )

    detected_rows = fold_event_metrics_df[fold_event_metrics_df["seizure_detected"] == 1]
    tiw = detected_rows["time_in_warning_sec"].dropna()
    if len(tiw) > 0:
        tiw_mean, tiw_median, tiw_min, tiw_max = (
            float(tiw.mean()), float(tiw.median()), float(tiw.min()), float(tiw.max())
        )
    else:
        tiw_mean = tiw_median = tiw_min = tiw_max = np.nan

    return {
        "n_folds": n_folds,
        "n_seizures_detected": n_detected,
        "seizure_sensitivity": seizure_sensitivity,
        "total_false_alarm_events": total_fa_events,
        "total_interictal_hours": total_interictal_hours,
        "false_alarms_per_hour": pooled_fa_per_hour,
        "n_detected_with_time_in_warning": int(len(tiw)),
        "time_in_warning_sec_mean": tiw_mean,
        "time_in_warning_sec_median": tiw_median,
        "time_in_warning_sec_min": tiw_min,
        "time_in_warning_sec_max": tiw_max,
    }


print("[OK] Metric functions reused verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.")


[OK] Metric functions reused verbatim from EEG_logistic_loso_binary.ipynb / EEG_seizure_prediction_metrics.ipynb.


## 4. Sequence-construction design (inspected from real metadata, not assumed)

**Metadata actually inspected before designing this** (not invented):
`labels_binary_preictal.parquet` has `recording_id` and an `epoch_idx` that
is verified (checked programmatically, see assertion below) to be
**contiguous with no gaps** within every one of the 17 recordings (0, 1, 2,
... in fixed 30s steps, `t_start_sec`/`t_end_sec` incrementing by exactly
`EPOCH_SEC` each step) -- i.e. the raw chronological epoch sequence per
recording is intact, including seizure-overlapping and excluded epochs.
`recording_id` is therefore both the natural chronology-continuity boundary
and the natural TRAIN/TEST-independence boundary (recordings are never
split across patients).

**History length:** the historical 4-class TCN
(`EEG_TCN_seizure_forecasting.ipynb`) used `SEQUENCE_LENGTH = 10` epochs
(10 x 30s = 5 minutes of history). Inspected as a starting reference (not
copied methodology -- its leakage/splitting/scaling approach is NOT reused,
only the epoch-count magnitude). A quick feasibility check (run for real,
not assumed) against the actual 57 LOSO folds under this notebook's stricter
same-split rule (below) found `SEQUENCE_LENGTH=10` still leaves 48/57 folds
(84%) with at least one valid TEST sequence, vs 54/57 at `SEQUENCE_LENGTH=5`.
**`SEQUENCE_LENGTH=10` (5 minutes) is used**, preserving continuity with the
historical reference and giving the TCN a genuinely useful temporal window,
at the documented cost of 9/57 folds losing TEST evaluability entirely
(Section 9-10 report exactly which folds and why).

**The rule enforced for every sequence (STRICTER than the historical
TCN's, per this task's explicit requirement):** a sequence for target epoch
`t` is valid only if epochs `t-9, t-8, ..., t` (10 consecutive `epoch_idx`
values):
1. belong to the same `recording_id` (never crosses a recording boundary),
2. are chronologically contiguous (`epoch_idx` increases by exactly 1 each
   step -- verified explicitly per sequence, not assumed from the global
   check above),
3. **all belong to the SAME fold split** (all in this fold's `train_idx`, or
   all in this fold's `test_idx`) -- so a sequence NEVER crosses a
   TRAIN/TEST boundary, and never silently includes an excluded/
   seizure-overlapping epoch (those belong to neither `train_idx` nor
   `test_idx`, so their presence anywhere in the window breaks the
   same-split run and the window is simply not built).

This is a materially stricter rule than the historical TCN's (which used a
single recording-level TRAIN/VAL/TEST split, so within-recording windows
never crossed a split boundary by construction, and history could include
seizure-overlapping epochs as raw context). Under this task's binary LOSO
folds, TRAIN and TEST can both draw epochs from the SAME recording (Section
20.3 of project_state.md -- recording-level fold assignment was rejected
precisely because some recordings contain 2 eligible seizures), so this
rule is a genuinely new, non-trivial construction, not a copy-paste.


In [4]:

# Verify the epoch_idx-contiguity claim above programmatically (not assumed).
_bad = 0
for _rid, _g in df.groupby("recording_id"):
    _g = _g.sort_values("epoch_idx")
    _idx = _g["epoch_idx"].to_numpy()
    if not (np.diff(_idx) == 1).all():
        _bad += 1
        print(f"[WARN] Non-contiguous epoch_idx in {_rid}")
assert _bad == 0, "epoch_idx is not contiguous in every recording -- sequence construction assumption violated"
print(f"[OK] epoch_idx is contiguous (no gaps) in all {df['recording_id'].nunique()} recordings -- verified, not assumed.")

SEQUENCE_LENGTH = 10  # 10 x 30s epochs = 5 minutes of history; see Section 4 rationale
print(f"SEQUENCE_LENGTH = {SEQUENCE_LENGTH} epochs ({SEQUENCE_LENGTH * EPOCH_SEC / 60:.1f} minutes of history)")


[OK] epoch_idx is contiguous (no gaps) in all 17 recordings -- verified, not assumed.
SEQUENCE_LENGTH = 10 epochs (5.0 minutes of history)


## 5. Sequence construction and sequence-level leakage assertions

`build_split_sequences` restricts a fold's existing `train_idx`/`test_idx`
(from the verbatim-reused `build_loso_fold`, Section 2 -- unchanged) to
valid same-split, same-recording, contiguous 10-epoch windows, per Section
4's rule. `verify_fold_sequences` then re-derives, independently, that every
built window obeys all of: same recording, contiguous `epoch_idx`, entirely
within one split, and (for TRAIN specifically) contains no epoch belonging
to the held-out seizure's PREICTAL context -- run for every fold, not
sampled.


In [5]:

def build_split_sequences(patient_df, scaled_df, scaled_cols, split_idx, lab_col, seq_len):
    '''Build (X, y, meta) sequences whose FULL window (history + target) lies
    entirely within `split_idx` (a fold's train_idx or test_idx), never
    crossing a recording boundary, with strictly contiguous epoch_idx.
    Returns X: (n_seq, seq_len, n_features) float32, y: (n_seq,) int64 (target
    epoch's PREICTAL/INTERICTAL label), meta: DataFrame with one row per
    sequence (orig_index, recording_id, epoch_idx, t_start_sec, t_end_sec,
    time_to_next_seizure_sec of the TARGET epoch, plus window_orig_indices
    for later leakage verification).'''
    split_set = set(split_idx)
    X_list, y_list, meta_rows = [], [], []
    for rid, g in patient_df.groupby("recording_id", sort=False):
        g = g.sort_values("epoch_idx")
        orig_idx = g.index.to_numpy()
        epoch_idx_arr = g["epoch_idx"].to_numpy()
        in_split = np.array([oi in split_set for oi in orig_idx])
        feats = scaled_df.loc[orig_idx, scaled_cols].to_numpy()
        n = len(orig_idx)
        run = 0
        for i in range(n):
            run = run + 1 if in_split[i] else 0
            if run >= seq_len:
                window_slice = slice(i - seq_len + 1, i + 1)
                window_orig = orig_idx[window_slice]
                window_epoch_idx = epoch_idx_arr[window_slice]
                assert (np.diff(window_epoch_idx) == 1).all(), "Non-contiguous window (should be impossible)"
                assert len(set(window_orig) - split_set) == 0, "Window contains an epoch outside this split (should be impossible)"
                target_orig = orig_idx[i]
                X_list.append(feats[window_slice, :])
                y_list.append(int(patient_df.loc[target_orig, lab_col] == "PREICTAL"))
                meta_rows.append({
                    "orig_index": target_orig,
                    "recording_id": rid,
                    "epoch_idx": patient_df.loc[target_orig, "epoch_idx"],
                    "t_start_sec": patient_df.loc[target_orig, "t_start_sec"],
                    "t_end_sec": patient_df.loc[target_orig, "t_end_sec"],
                    "time_to_next_seizure_sec": patient_df.loc[target_orig, "time_to_next_seizure_sec"],
                    "window_orig_indices": tuple(window_orig),
                })
    n_feat = len(scaled_cols)
    if len(X_list) == 0:
        return (np.zeros((0, seq_len, n_feat), dtype=np.float32), np.zeros((0,), dtype=np.int64),
                pd.DataFrame(columns=["orig_index", "recording_id", "epoch_idx", "t_start_sec",
                                       "t_end_sec", "time_to_next_seizure_sec", "window_orig_indices"]))
    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.int64)
    meta = pd.DataFrame(meta_rows)
    return X, y, meta


def verify_fold_sequences(train_idx, test_idx, meta_train, meta_test, patient_df, lab_col, held_out_seizure_id):
    '''Independent, per-fold leakage re-verification for sequences (beyond validate_loso_fold's epoch-level checks).'''
    train_set, test_set = set(train_idx), set(test_idx)

    all_train_window_idx = set()
    for w in meta_train.get("window_orig_indices", []):
        all_train_window_idx |= set(w)
    all_test_window_idx = set()
    for w in meta_test.get("window_orig_indices", []):
        all_test_window_idx |= set(w)

    assert all_train_window_idx <= train_set, "A TRAIN sequence window used an epoch outside train_idx"
    assert all_test_window_idx <= test_set, "A TEST sequence window used an epoch outside test_idx"
    assert all_train_window_idx.isdisjoint(all_test_window_idx), \
        "A TRAIN sequence window and a TEST sequence window share an epoch -- TRAIN/TEST boundary crossed"

    held_out_preictal_idx = set(
        patient_df.index[(patient_df[lab_col] == "PREICTAL") & (patient_df["next_seizure_id"] == held_out_seizure_id)]
    )
    assert all_train_window_idx.isdisjoint(held_out_preictal_idx), \
        "Held-out seizure's PREICTAL epoch(s) leaked into a TRAIN sequence window"

    if len(meta_train):
        assert meta_train["recording_id"].map(lambda r: True).all()  # recording_id present, sanity
    return True


print("[OK] Sequence-construction and sequence-level leakage-verification functions defined.")


[OK] Sequence-construction and sequence-level leakage-verification functions defined.


## 6. TCN architecture

A compact causal, dilated TCN -- fresh implementation for this task,
architecturally inspired by (not copied from) the historical
`EEG_TCN_seizure_forecasting.ipynb`'s `TinyTCN`/`CausalConv1d` (credited
there as the reference design: 3 dilated causal-conv blocks, dilation
1/2/4). Adapted here for a **binary** single-logit output (this task's
target is PREICTAL-vs-INTERICTAL, not the old 4-class target) and applied
under the new LOSO/patient-specific-scaling methodology, not the old
notebook's pooled/recording-level one.

`CausalConv1d` pads only on the LEFT (past) side by `(kernel_size-1) *
dilation`, so output timestep `t` is a function of input timesteps `<= t`
only -- no future information ever reaches the representation at `t`,
satisfying the "no future information" requirement structurally (by
construction, not just by convention). The final (rightmost / most recent)
timestep's hidden representation -- corresponding to the target epoch `t`
itself, with the causal receptive field covering `t-9..t` -- is fed to a
single linear output unit (one logit; `sigmoid(logit) = P(PREICTAL)`).

Deliberately small: `hidden=16` channels, `kernel_size=3`, 3 blocks -- a few
thousand parameters, appropriate for the smallest per-fold TRAIN sets here
(as few as ~31 sequences for some CHB12 folds, Section 9). No architecture
search was performed; this single fixed configuration is used for every
fold.


In [6]:

import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalConv1d(nn.Module):
    '''A Conv1d that only ever looks BACKWARD in time (causal), never forward.
    Pads only on the left (past) side by (kernel_size - 1) * dilation, so
    output[t] is a function of input[<= t] only.'''

    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.left_pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=0, dilation=dilation)

    def forward(self, x):
        x = F.pad(x, (self.left_pad, 0))
        return self.conv(x)


class TinyTCNBinary(nn.Module):
    '''Compact 3-block causal, dilated TCN with a single binary output logit.
    Input : (batch, seq_len, n_features) -- as produced by build_split_sequences.
    Output: (batch,) raw logits; sigmoid(logit) = P(PREICTAL).'''

    def __init__(self, n_features, hidden=16, kernel_size=3):
        super().__init__()
        self.block1 = CausalConv1d(n_features, hidden, kernel_size, dilation=1)
        self.block2 = CausalConv1d(hidden, hidden, kernel_size, dilation=2)
        self.block3 = CausalConv1d(hidden, hidden, kernel_size, dilation=4)
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, x):
        x = x.transpose(1, 2)  # (batch, features, time) for Conv1d
        x = self.relu(self.block1(x))
        x = self.relu(self.block2(x))
        x = self.relu(self.block3(x))
        last = x[:, :, -1]  # representation AT the target epoch t (causal receptive field t-9..t)
        return self.classifier(last).squeeze(-1)


TCN_HIDDEN = 16
TCN_KERNEL_SIZE = 3
_n_params = sum(p.numel() for p in TinyTCNBinary(len(FEATURE_COLS), TCN_HIDDEN, TCN_KERNEL_SIZE).parameters())
print(f"[OK] TinyTCNBinary defined: hidden={TCN_HIDDEN}, kernel_size={TCN_KERNEL_SIZE}, "
      f"dilations=(1,2,4), n_features={len(FEATURE_COLS)} -> {_n_params} trainable parameters.")


[OK] TinyTCNBinary defined: hidden=16, kernel_size=3, dilations=(1,2,4), n_features=14 -> 2273 trainable parameters.


## 7. TRAIN-only validation split (for early stopping)

TEST is never touched before final fold evaluation. A validation subset is
carved **only from this fold's TRAIN sequences**, used exclusively for
early-stopping the TCN's own training. Two documented strategies, chosen
per fold based on how many distinct recordings actually contribute TRAIN
sequences (this varies fold-to-fold and is reported live, not assumed):

1. **Recording-level holdout (used when TRAIN sequences span >= 2
   recordings, the common case)**: whole recordings are assigned to
   validation (smallest first) until close to 20% of TRAIN sequences are
   covered, always leaving at least one recording for training. Because
   sequences never cross recording boundaries (Section 4), this split is
   leakage-free by construction -- no sequence window can straddle the
   train/validation divide.
2. **Chronological gap fallback (used only when all of a fold's TRAIN
   sequences come from a single recording)**: the last ~20% of sequences
   (by target `epoch_idx`) become validation; TRAIN sequences whose window
   would overlap the validation region (within `SEQUENCE_LENGTH-1` epochs of
   the split point) are dropped entirely rather than risk window overlap.

If neither split can be constructed with both classes representable in the
resulting sub-TRAIN set (e.g. extremely small folds), the fold falls back to
training on the full TRAIN set for a small fixed number of epochs with no
early stopping -- documented explicitly per fold, not silently substituted.


In [7]:

def split_train_val_by_recording(meta_train, val_frac, seq_len):
    n_total = len(meta_train)
    recs = meta_train["recording_id"].unique().tolist()
    if len(recs) >= 2:
        counts = meta_train.groupby("recording_id").size().sort_values()
        target = max(1, round(val_frac * n_total))
        val_recs, running = [], 0
        for rid, cnt in counts.items():
            if len(val_recs) >= len(counts) - 1:
                break
            val_recs.append(rid)
            running += int(cnt)
            if running >= target:
                break
        val_mask = meta_train["recording_id"].isin(val_recs).to_numpy()
        drop_mask = np.zeros(n_total, dtype=bool)
        method = f"recording-level holdout: {len(val_recs)}/{len(recs)} recordings as validation {val_recs} ({running}/{n_total} sequences)"
        return val_mask, drop_mask, method
    else:
        g = meta_train.sort_values("epoch_idx").reset_index(drop=True)
        n_val = max(1, round(val_frac * n_total))
        val_start = n_total - n_val
        gap = seq_len - 1
        drop_start = max(0, val_start - gap)
        sorted_orig = g["orig_index"].to_numpy()
        val_orig = set(sorted_orig[val_start:])
        drop_orig = set(sorted_orig[drop_start:val_start])
        val_mask = meta_train["orig_index"].isin(val_orig).to_numpy()
        drop_mask = meta_train["orig_index"].isin(drop_orig).to_numpy()
        method = (f"single-recording chronological holdout: last {n_val}/{n_total} sequences as validation, "
                  f"{len(drop_orig)} boundary sequence(s) dropped to enforce a {gap}-epoch non-overlap gap")
        return val_mask, drop_mask, method


print("[OK] TRAIN-only validation split function defined.")


[OK] TRAIN-only validation split function defined.


## 8. Training procedure

Recorded here explicitly, exactly as required:

- **Architecture**: `TinyTCNBinary` (Section 6), `hidden=16`,
  `kernel_size=3`, dilations `(1, 2, 4)`.
- **Sequence length**: 10 epochs (5 minutes), Section 4.
- **Optimizer**: Adam, learning rate `1e-3`, no weight decay.
- **Loss**: `BCEWithLogitsLoss`, with `pos_weight = n_train_neg / n_train_pos`
  computed from the exact sequences passed to `.fit()` (the sub-TRAIN split
  after validation carve-out, Section 7) -- the TCN analogue of LR's/RF's
  `class_weight="balanced"`, computed strictly from TRAIN data.
- **Batch size**: `min(32, n_subtrain)` (never larger than the fold's own
  sub-TRAIN set).
- **Epochs**: up to 60, with early stopping (patience 10 epochs, minimum
  improvement `1e-4` on validation BCE loss) when a validation split is
  available; otherwise a fixed 30 epochs with no early stopping (documented
  fallback case).
- **Random seeds**: `torch.manual_seed(42)` and `np.random.seed(42)` reset
  identically before every fold's model is initialized and trained -- each
  fold gets an independently-seeded but fully reproducible run.
- **Decision threshold**: 0.5 (fixed, not tuned on TEST) -- identical to LR
  and RF.

No hyperparameter (architecture size, learning rate, batch size, epoch
budget, patience) was tuned per fold or against TEST data; this single
configuration is applied to every fold.


In [8]:

TCN_MAX_EPOCHS = 60
TCN_PATIENCE = 10
TCN_MIN_DELTA = 1e-4
TCN_LR = 1e-3
TCN_BATCH_SIZE = 32
TCN_FALLBACK_EPOCHS = 30  # used only when no validation split is available (no early stopping)
VAL_FRACTION = 0.20

print(f"TCN_MAX_EPOCHS={TCN_MAX_EPOCHS}, TCN_PATIENCE={TCN_PATIENCE}, TCN_MIN_DELTA={TCN_MIN_DELTA}, "
      f"TCN_LR={TCN_LR}, TCN_BATCH_SIZE={TCN_BATCH_SIZE}, TCN_FALLBACK_EPOCHS={TCN_FALLBACK_EPOCHS}, "
      f"VAL_FRACTION={VAL_FRACTION}")


def train_tcn_fold(X_subtrain, y_subtrain, X_val, y_val, n_features, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = TinyTCNBinary(n_features, TCN_HIDDEN, TCN_KERNEL_SIZE)
    opt = torch.optim.Adam(model.parameters(), lr=TCN_LR)

    n_pos = int(y_subtrain.sum())
    n_neg = int(len(y_subtrain) - n_pos)
    if n_pos == 0 or n_neg == 0:
        raise RuntimeError(f"sub-TRAIN has only one class (pos={n_pos}, neg={n_neg}) -- cannot fit.")
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    X_subtrain_t = torch.from_numpy(X_subtrain)
    y_subtrain_t = torch.from_numpy(y_subtrain.astype(np.float32))
    has_val = X_val is not None and len(X_val) > 0 and len(np.unique(y_val)) >= 1
    if has_val:
        X_val_t = torch.from_numpy(X_val)
        y_val_t = torch.from_numpy(y_val.astype(np.float32))

    n = len(X_subtrain_t)
    bs = max(1, min(TCN_BATCH_SIZE, n))
    max_epochs = TCN_MAX_EPOCHS if has_val else TCN_FALLBACK_EPOCHS

    best_val, best_state, epochs_no_improve = np.inf, None, 0
    n_epochs_run = 0
    for epoch in range(max_epochs):
        model.train()
        perm = torch.randperm(n)
        for start in range(0, n, bs):
            batch_idx = perm[start:start + bs]
            xb, yb = X_subtrain_t[batch_idx], y_subtrain_t[batch_idx]
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
        n_epochs_run = epoch + 1
        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = loss_fn(model(X_val_t), y_val_t).item()
            if val_loss < best_val - TCN_MIN_DELTA:
                best_val = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
            if epochs_no_improve >= TCN_PATIENCE:
                break
    if has_val and best_state is not None:
        model.load_state_dict(best_state)
    return model, n_epochs_run, has_val, (float(best_val) if has_val else np.nan)


print("[OK] Training function defined.")


TCN_MAX_EPOCHS=60, TCN_PATIENCE=10, TCN_MIN_DELTA=0.0001, TCN_LR=0.001, TCN_BATCH_SIZE=32, TCN_FALLBACK_EPOCHS=30, VAL_FRACTION=0.2
[OK] Training function defined.


## 9. Main experiment loop

Same 57 (horizon, patient, held-out-seizure) folds as LR/RF, built with the
identical `build_loso_fold`. For each fold: build TRAIN and TEST sequences
(Section 5), run the sequence-level leakage re-verification, and:
- if TRAIN sequences have fewer than 2 classes -> **should not happen**
  (verified separately, Section 4/10) -- raise, do not silently skip.
- if TEST has **zero** valid sequences -> **skip modeling for this fold**
  (cannot evaluate it) and record it in `skipped_folds` with the reason,
  rather than fabricating a result.
- otherwise: split TRAIN into sub-TRAIN/validation (Section 7), train the
  TCN (Section 8) and a `DummyClassifier(strategy="most_frequent")` (fit on
  the FULL fold TRAIN sequences, matching LR/RF's dummy baseline
  convention), predict on TEST, and record predictions + per-fold
  diagnostics (sequence counts, validation method used, epochs trained).


In [9]:

seizure_inventory = {}
for pid in PATIENTS:
    pdf = df[df[PATIENT_COL] == pid]
    seizure_inventory[pid] = sorted(pdf["seizure_id"].dropna().unique())

prediction_rows = []
fold_diagnostic_rows = []
skipped_folds = []
n_folds_attempted = 0
n_folds_evaluated = 0

for H in HORIZONS:
    fold_context = assign_fold_context(df, H)
    lab_col = f"binary_label_h{H}"

    for pid in PATIENTS:
        patient_df = df[df[PATIENT_COL] == pid].copy()
        assert patient_df["patient_id"].nunique() == 1 and patient_df["patient_id"].iloc[0] == pid

        eligible, _ineligible = compute_seizure_eligibility(df, pid, H, seizure_inventory[pid])

        for fold_id, sid in enumerate(eligible, start=1):
            n_folds_attempted += 1
            fold = build_loso_fold(patient_df, H, sid, fold_context, FEATURE_COLS)
            validate_loso_fold(fold, patient_df, FEATURE_COLS)  # same 9 epoch-level leakage checks as LR/RF

            train_idx, test_idx = fold["train_idx"], fold["test_idx"]
            scaled_df, scaled_cols = fold["scaled_df"], fold["scaled_cols"]
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].nunique() == 1
            assert scaled_df.loc[train_idx.union(test_idx), "patient_id"].iloc[0] == pid

            X_train, y_train, meta_train = build_split_sequences(
                patient_df, scaled_df, scaled_cols, train_idx, lab_col, SEQUENCE_LENGTH)
            X_test, y_test, meta_test = build_split_sequences(
                patient_df, scaled_df, scaled_cols, test_idx, lab_col, SEQUENCE_LENGTH)

            verify_fold_sequences(train_idx, test_idx, meta_train, meta_test, patient_df, lab_col, sid)

            n_train_pos, n_train_neg = int(y_train.sum()), int(len(y_train) - y_train.sum())
            fold_row = {
                "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id, "held_out_seizure_id": sid,
                "n_train_epochs": len(train_idx), "n_test_epochs": len(test_idx),
                "n_train_seq": len(y_train), "n_test_seq": len(y_test),
                "train_seq_preictal": n_train_pos, "train_seq_interictal": n_train_neg,
            }

            if len(y_train) == 0 or n_train_pos == 0 or n_train_neg == 0:
                fold_row["status"] = "skipped_train_single_class_or_empty"
                skipped_folds.append(fold_row)
                fold_diagnostic_rows.append(fold_row)
                continue
            if len(y_test) == 0:
                fold_row["status"] = "skipped_zero_test_sequences"
                skipped_folds.append(fold_row)
                fold_diagnostic_rows.append(fold_row)
                continue

            val_mask, drop_mask, val_method = split_train_val_by_recording(meta_train, VAL_FRACTION, SEQUENCE_LENGTH)
            keep_mask = (~val_mask) & (~drop_mask)
            X_subtrain, y_subtrain = X_train[keep_mask], y_train[keep_mask]
            X_val, y_val = X_train[val_mask], y_train[val_mask]

            use_val = (
                len(X_subtrain) >= 4 and len(np.unique(y_subtrain)) == 2
                and len(X_val) >= 1
            )
            if not use_val:
                X_subtrain, y_subtrain = X_train, y_train
                X_val, y_val = None, None
                val_method = "NONE (fallback: fold too small for a validation split -- trained on full TRAIN, "\
                             f"fixed {TCN_FALLBACK_EPOCHS} epochs, no early stopping)"

            model, n_epochs_run, used_val, best_val_loss = train_tcn_fold(
                X_subtrain, y_subtrain, X_val, y_val, len(scaled_cols), seed=RANDOM_STATE)

            dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
            dummy.fit(X_train.reshape(len(X_train), -1), y_train)  # flattened; DummyClassifier ignores X content anyway

            model.eval()
            with torch.no_grad():
                test_logits = model(torch.from_numpy(X_test))
                y_prob_tcn = torch.sigmoid(test_logits).numpy()
                train_logits = model(torch.from_numpy(X_train))  # FULL fold TRAIN (not just sub-train) -- overfitting diagnostic only, never used for model selection
                y_pred_train_tcn = (torch.sigmoid(train_logits).numpy() >= THRESHOLD).astype(int)
            train_accuracy_tcn = float((y_pred_train_tcn == y_train).mean())
            y_pred_tcn = (y_prob_tcn >= THRESHOLD).astype(int)

            pos_idx_dummy = list(dummy.classes_).index(1) if 1 in dummy.classes_ else None
            if pos_idx_dummy is not None:
                y_prob_dummy = dummy.predict_proba(X_test.reshape(len(X_test), -1))[:, pos_idx_dummy]
            else:
                y_prob_dummy = np.zeros(len(X_test))
            y_pred_dummy = (y_prob_dummy >= THRESHOLD).astype(int)

            for model_name, y_prob, y_pred in [("tcn", y_prob_tcn, y_pred_tcn), ("dummy_most_frequent", y_prob_dummy, y_pred_dummy)]:
                rows = meta_test[["recording_id", "epoch_idx", "t_start_sec", "t_end_sec", "time_to_next_seizure_sec"]].reset_index(drop=True).copy()
                rows["patient_id"] = pid
                rows["horizon_minutes"] = H
                rows["fold_id"] = fold_id
                rows["held_out_seizure_id"] = sid
                rows["model"] = model_name
                rows["y_true"] = y_test
                rows["y_prob"] = y_prob
                rows["y_pred"] = y_pred
                prediction_rows.append(rows)

            fold_row.update({
                "status": "evaluated",
                "n_subtrain_seq": len(X_subtrain), "n_val_seq": (len(X_val) if X_val is not None else 0),
                "validation_method": val_method, "used_early_stopping": bool(used_val),
                "n_epochs_trained": int(n_epochs_run), "best_val_loss": best_val_loss,
                "train_accuracy": train_accuracy_tcn,
            })
            fold_diagnostic_rows.append(fold_row)
            n_folds_evaluated += 1

predictions_df = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame(columns=REQUIRED_PRED_COLS + ["model"])
fold_diagnostics_df = pd.DataFrame(fold_diagnostic_rows)
skipped_df = pd.DataFrame(skipped_folds)

print(f"[OK] Attempted {n_folds_attempted} LOSO folds (expected 57).")
print(f"[OK] Evaluated (TCN trained + predicted on >=1 TEST sequence): {n_folds_evaluated}/{n_folds_attempted}")
print(f"[INFO] Skipped folds (insufficient sequence history, not silently substituted): {len(skipped_df)}")
if len(skipped_df):
    print(skipped_df[["patient_id", "horizon_minutes", "held_out_seizure_id", "status", "n_train_seq", "n_test_seq"]].to_string(index=False))
assert n_folds_attempted == 57, f"Expected 57 LOSO folds, attempted {n_folds_attempted}"
print(f"\npredictions_df: {predictions_df.shape}")


[OK] Attempted 57 LOSO folds (expected 57).
[OK] Evaluated (TCN trained + predicted on >=1 TEST sequence): 48/57
[INFO] Skipped folds (insufficient sequence history, not silently substituted): 9
patient_id  horizon_minutes held_out_seizure_id                      status  n_train_seq  n_test_seq
     chb12                5     chb_s12_r23_sz0 skipped_zero_test_sequences           32           0
     chb24                5     chb_s24_r01_sz1 skipped_zero_test_sequences          277           0
     chb24                5     chb_s24_r03_sz0 skipped_zero_test_sequences          277           0
     chb12               15     chb_s12_r23_sz0 skipped_zero_test_sequences          105           0
     chb24               15     chb_s24_r01_sz1 skipped_zero_test_sequences          479           0
     chb24               15     chb_s24_r03_sz0 skipped_zero_test_sequences          479           0
     chb12               30     chb_s12_r23_sz0 skipped_zero_test_sequences          168          

## 9b. Cross-check epoch-level fold construction against the saved LOSO manifest

Same cross-check LR and RF each performed: the epoch-level `train_idx`/
`test_idx` sizes recorded per fold above (before any sequence restriction)
must match `outputs/loso_fold_manifest.parquet` exactly -- confirms this
notebook's fold construction (Section 2, verbatim-reused code) produced the
same underlying folds as LR and RF, before sequence construction narrows
them further.


In [10]:

manifest_path = output_dir / "loso_fold_manifest.parquet"
existing_manifest = pd.read_parquet(manifest_path)
manifest_check = fold_diagnostics_df.merge(
    existing_manifest, on=["patient_id", "horizon_minutes", "held_out_seizure_id"], suffixes=("_tcn", "_manifest")
)
assert len(manifest_check) == len(fold_diagnostics_df) == len(existing_manifest) == 57
assert (manifest_check["n_train_epochs"] == manifest_check["train_total"]).all()
assert (manifest_check["n_test_epochs"] == manifest_check["test_total"]).all()
print(f"[OK] All 57 folds' epoch-level TRAIN/TEST sizes match outputs/loso_fold_manifest.parquet exactly -- "
      f"same underlying LOSO folds as LR and RF, confirmed independently in this notebook (not assumed from "
      f"code identity alone).")

lr_fold_metrics = pd.read_csv(lr_run_dir / "fold_metrics.csv")
lr_fold_metrics = lr_fold_metrics[lr_fold_metrics["model"] == "logistic_regression"].copy()
lr_fold_metrics["n_test_epochs_lr"] = lr_fold_metrics["n_test_preictal"] + lr_fold_metrics["n_test_interictal"]
cross_lr = fold_diagnostics_df.merge(
    lr_fold_metrics[["patient_id", "horizon_minutes", "held_out_seizure_id", "n_train", "n_test_epochs_lr"]],
    on=["patient_id", "horizon_minutes", "held_out_seizure_id"],
)
assert len(cross_lr) == 57
assert (cross_lr["n_train_epochs"] == cross_lr["n_train"]).all()
assert (cross_lr["n_test_epochs"] == cross_lr["n_test_epochs_lr"]).all()
print(f"[OK] All 57 folds' epoch-level TRAIN/TEST sizes also match the LR run's own saved fold_metrics.csv "
      f"exactly -- this notebook, LR, and RF all evaluate the identical underlying LOSO folds; only TCN's "
      f"further sequence-construction restriction (Section 4-5, 9-10) differs, and is reported explicitly.")


[OK] All 57 folds' epoch-level TRAIN/TEST sizes match outputs/loso_fold_manifest.parquet exactly -- same underlying LOSO folds as LR and RF, confirmed independently in this notebook (not assumed from code identity alone).
[OK] All 57 folds' epoch-level TRAIN/TEST sizes also match the LR run's own saved fold_metrics.csv exactly -- this notebook, LR, and RF all evaluate the identical underlying LOSO folds; only TCN's further sequence-construction restriction (Section 4-5, 9-10) differs, and is reported explicitly.


## 10. Sequence/fold coverage report

Explicit, honest accounting of TCN's coverage relative to LR/RF's full
57/57 -- required because the stricter same-split sequence rule (Section 4)
means not every LOSO fold can produce a valid TEST sequence.


In [11]:

lr_folds_57 = 57
rf_folds_57 = 57
print("=" * 70)
print("FOLD COVERAGE: TCN vs LR/RF")
print("=" * 70)
print(f"LR:  57/57 folds evaluated (epoch-level, no sequence-history requirement).")
print(f"RF:  57/57 folds evaluated (epoch-level, no sequence-history requirement).")
print(f"TCN: {n_folds_evaluated}/57 folds evaluated (sequence-level: needs {SEQUENCE_LENGTH} contiguous "
      f"same-split epochs for >=1 TEST sequence).")
print(f"\nPer-horizon TCN coverage (of the same 19 eligible seizures LR/RF use):")
cov = fold_diagnostics_df.groupby("horizon_minutes")["status"].value_counts().unstack(fill_value=0)
print(cov.to_string())
print(f"\n[LIMITATION] TCN's seizure-sensitivity/FA-hour/AUC-PR numbers below are therefore computed over "
      f"{n_folds_evaluated} of 57 total fold-rows, NOT the full 57 -- this is reported explicitly throughout, "
      f"never silently padded to look like full coverage.")


FOLD COVERAGE: TCN vs LR/RF
LR:  57/57 folds evaluated (epoch-level, no sequence-history requirement).
RF:  57/57 folds evaluated (epoch-level, no sequence-history requirement).
TCN: 48/57 folds evaluated (sequence-level: needs 10 contiguous same-split epochs for >=1 TEST sequence).

Per-horizon TCN coverage (of the same 19 eligible seizures LR/RF use):
status           evaluated  skipped_zero_test_sequences
horizon_minutes                                        
5                       16                            3
15                      16                            3
30                      16                            3

[LIMITATION] TCN's seizure-sensitivity/FA-hour/AUC-PR numbers below are therefore computed over 48 of 57 total fold-rows, NOT the full 57 -- this is reported explicitly throughout, never silently padded to look like full coverage.


## 11. Metrics: per-fold, then pooled (identical framework to LR/RF)


In [12]:

fold_metrics_rows = []
for (pid, H, fold_id, sid, model_name), fold_df in predictions_df.groupby(
    ["patient_id", "horizon_minutes", "fold_id", "held_out_seizure_id", "model"]
):
    event_m = fold_seizure_event_metrics(fold_df)
    epoch_m = fold_epoch_metrics(fold_df)
    fold_metrics_rows.append({
        "patient_id": pid, "horizon_minutes": H, "fold_id": fold_id,
        "held_out_seizure_id": sid, "model": model_name,
        **event_m, **epoch_m,
    })
fold_metrics_df = pd.DataFrame(fold_metrics_rows)
print(f"fold_metrics_df: {fold_metrics_df.shape}")

def summarize_group(event_df, epoch_pool_df, n_folds_in_group):
    ev = pool_event_metrics(event_df)
    ep = pool_epoch_metrics(epoch_pool_df)
    return {"n_folds_evaluated": n_folds_in_group, **ev, **ep}

patient_horizon_rows = []
for model_name in ["tcn", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for pid in PATIENTS:
        for H in HORIZONS:
            ev_subset = fm_model[(fm_model["patient_id"] == pid) & (fm_model["horizon_minutes"] == H)]
            ep_subset = pred_model[(pred_model["patient_id"] == pid) & (pred_model["horizon_minutes"] == H)]
            if len(ev_subset) == 0:
                continue
            row = {"model": model_name, "patient_id": pid, "horizon_minutes": H}
            row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
            patient_horizon_rows.append(row)
patient_horizon_summary_df = pd.DataFrame(patient_horizon_rows)

overall_horizon_rows = []
for model_name in ["tcn", "dummy_most_frequent"]:
    fm_model = fold_metrics_df[fold_metrics_df["model"] == model_name]
    pred_model = predictions_df[predictions_df["model"] == model_name]
    for H in HORIZONS:
        ev_subset = fm_model[fm_model["horizon_minutes"] == H]
        ep_subset = pred_model[pred_model["horizon_minutes"] == H]
        n_eligible_at_H = len(compute_seizure_eligibility(df, "chb12", H, seizure_inventory["chb12"])[0]) + \
                           len(compute_seizure_eligibility(df, "chb24", H, seizure_inventory["chb24"])[0])
        row = {"model": model_name, "horizon_minutes": H, "n_eligible_seizures": n_eligible_at_H}
        row.update(summarize_group(ev_subset, ep_subset, len(ev_subset)))
        overall_horizon_rows.append(row)
overall_horizon_summary_df = pd.DataFrame(overall_horizon_rows)

pd.set_option("display.width", 160)
display_cols = ["model", "patient_id", "horizon_minutes", "n_folds_evaluated", "seizure_sensitivity",
                 "false_alarms_per_hour", "auc_pr", "f1_preictal", "accuracy"]
print("=== PER PATIENT + HORIZON (TCN, coverage-aware) ===")
print(patient_horizon_summary_df[display_cols].to_string(index=False))

print("\n=== OVERALL BY HORIZON (pooled across chb12 + chb24, TCN) ===")
overall_display_cols = ["model", "horizon_minutes", "n_eligible_seizures", "n_folds_evaluated", "seizure_sensitivity",
                         "false_alarms_per_hour", "auc_pr", "f1_preictal", "accuracy"]
print(overall_horizon_summary_df[overall_display_cols].to_string(index=False))
print("\n[NOTE] 'n_eligible_seizures' (19 per horizon, matching LR/RF) vs 'n_folds_evaluated' (TCN's actual "
      "coverage, Section 10) are DIFFERENT numbers here, unlike LR/RF where they coincide (57/57).")


fold_metrics_df: (96, 21)


=== PER PATIENT + HORIZON (TCN, coverage-aware) ===
              model patient_id  horizon_minutes  n_folds_evaluated  seizure_sensitivity  false_alarms_per_hour   auc_pr  f1_preictal  accuracy
                tcn      chb12                5                  4             1.000000                    NaN      NaN     1.000000  1.000000
                tcn      chb12               15                  4             1.000000                    NaN      NaN     1.000000  1.000000
                tcn      chb12               30                  4             1.000000                    NaN      NaN     0.955224  0.914286
                tcn      chb24                5                 12             0.916667               3.157895 0.064003     0.076389  0.039711
                tcn      chb24               15                 12             0.916667               3.157895 0.339207     0.591176  0.419624
                tcn      chb24               30                 12             0.916667   

## 12. Sanity checks: predicted-positive rate, confusion matrices, overfitting diagnostic

In [13]:

print("=== TEST-set predicted-positive rate (TCN, evaluated folds only) ===")
pos_rate_rows = []
for model_name in ["tcn", "dummy_most_frequent"]:
    for H in HORIZONS:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        if len(sub) == 0:
            continue
        pos_rate_rows.append({"model": model_name, "horizon_minutes": H,
                               "predicted_positive_rate": sub["y_pred"].mean(),
                               "true_positive_rate": sub["y_true"].mean(), "n": len(sub)})
pos_rate_df = pd.DataFrame(pos_rate_rows)
print(pos_rate_df.to_string(index=False))

flagged = pos_rate_df[(pos_rate_df["model"] == "tcn") & ((pos_rate_df["predicted_positive_rate"] < 0.02) | (pos_rate_df["predicted_positive_rate"] > 0.98))]
if len(flagged):
    print("\n[FLAG] TCN predicts essentially only one class at these horizons:")
    print(flagged.to_string(index=False))
else:
    print("\n[OK] TCN's predicted-positive rate is not degenerate (not <2% or >98%) at any horizon.")

print("\n=== Confusion matrices (pooled TEST sequences, per patient + horizon, TCN) ===")
for pid in PATIENTS:
    for H in HORIZONS:
        sub = predictions_df[(predictions_df["model"] == "tcn") & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)]
        if len(sub) == 0:
            print(f"{pid} H={H:2d}min:  (no evaluated folds -- see Section 10)")
            continue
        cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        print(f"{pid} H={H:2d}min:  TN={tn:4d}  FP={fp:4d}  FN={fn:4d}  TP={tp:4d}")

print("\n=== Overfitting diagnostic: mean fold TRAIN accuracy vs pooled TEST accuracy (TCN) ===")
evaluated_fd = fold_diagnostics_df[fold_diagnostics_df.status == "evaluated"]
train_acc_summary = (
    evaluated_fd.groupby(["patient_id", "horizon_minutes"])["train_accuracy"]
    .mean().reset_index().rename(columns={"train_accuracy": "mean_train_accuracy"})
)
test_acc_summary = patient_horizon_summary_df[patient_horizon_summary_df.model == "tcn"][
    ["patient_id", "horizon_minutes", "accuracy"]
].rename(columns={"accuracy": "pooled_test_accuracy"})
overfit_df = train_acc_summary.merge(test_acc_summary, on=["patient_id", "horizon_minutes"], how="left")
overfit_df["gap"] = overfit_df["mean_train_accuracy"] - overfit_df["pooled_test_accuracy"]
print(overfit_df.to_string(index=False))
print(f"\nMean epochs trained per fold: {fold_diagnostics_df.loc[fold_diagnostics_df.status=='evaluated','n_epochs_trained'].mean():.1f} "
      f"(max allowed {TCN_MAX_EPOCHS}, early-stopping patience {TCN_PATIENCE}); "
      f"folds using early stopping (validation available): "
      f"{int(fold_diagnostics_df.loc[fold_diagnostics_df.status=='evaluated','used_early_stopping'].sum())}/{n_folds_evaluated}")


=== TEST-set predicted-positive rate (TCN, evaluated folds only) ===
              model  horizon_minutes  predicted_positive_rate  true_positive_rate   n
                tcn                5                 1.000000            0.053381 281
                tcn               15                 0.978417            0.521583 556
                tcn               30                 0.986547            0.701794 892
dummy_most_frequent                5                 0.000000            0.053381 281
dummy_most_frequent               15                 0.138489            0.521583 556
dummy_most_frequent               30                 1.000000            0.701794 892

[FLAG] TCN predicts essentially only one class at these horizons:
model  horizon_minutes  predicted_positive_rate  true_positive_rate   n
  tcn                5                 1.000000            0.053381 281
  tcn               30                 0.986547            0.701794 892

=== Confusion matrices (pooled TEST sequences

## 13. Diagnostic plots

In [14]:

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    for model_name, style in [("tcn", "-"), ("dummy_most_frequent", "--")]:
        sub = predictions_df[(predictions_df["model"] == model_name) & (predictions_df["horizon_minutes"] == H)]
        if len(sub) == 0 or sub["y_true"].nunique() < 2:
            continue
        precision, recall, _ = precision_recall_curve(sub["y_true"], sub["y_prob"])
        ap = average_precision_score(sub["y_true"], sub["y_prob"])
        ax.plot(recall, precision, style, label=f"{model_name} (AP={ap:.3f})")
    ax.set_title(f"H = {H} min (pooled, evaluated folds only)")
    ax.set_xlabel("Recall")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Precision")
fig.suptitle("Precision-Recall curves -- pooled TEST sequences, TCN")
fig.tight_layout()
fig.savefig(plot_dir / "pr_curves_overall_by_horizon.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(len(PATIENTS), len(HORIZONS), figsize=(4 * len(HORIZONS), 4 * len(PATIENTS)))
for i, pid in enumerate(PATIENTS):
    for j, H in enumerate(HORIZONS):
        ax = axes[i, j]
        sub = predictions_df[(predictions_df["model"] == "tcn") & (predictions_df["patient_id"] == pid) & (predictions_df["horizon_minutes"] == H)]
        if len(sub) == 0:
            ax.text(0.5, 0.5, "no evaluated\nfolds", ha="center", va="center")
            ax.set_xticks([]); ax.set_yticks([])
        else:
            cm = confusion_matrix(sub["y_true"], sub["y_pred"], labels=[0, 1])
            ax.imshow(cm, cmap="Oranges")
            for r in range(2):
                for c in range(2):
                    ax.text(c, r, str(cm[r, c]), ha="center", va="center")
            ax.set_xticks([0, 1]); ax.set_xticklabels(["INTERICTAL", "PREICTAL"])
            ax.set_yticks([0, 1]); ax.set_yticklabels(["INTERICTAL", "PREICTAL"])
        ax.set_title(f"{pid}, H={H}min")
        if j == 0:
            ax.set_ylabel("True")
        if i == len(PATIENTS) - 1:
            ax.set_xlabel("Predicted")
fig.suptitle("Confusion matrices (pooled TEST sequences, TCN)")
fig.tight_layout()
fig.savefig(plot_dir / "confusion_matrices_tcn.png", dpi=150)
plt.close(fig)

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 4), sharey=True)
for ax, H in zip(axes, HORIZONS):
    sub = predictions_df[(predictions_df["model"] == "tcn") & (predictions_df["horizon_minutes"] == H)]
    if len(sub):
        ax.hist(sub.loc[sub["y_true"] == 0, "y_prob"], bins=20, alpha=0.6, label="true INTERICTAL", density=True)
        ax.hist(sub.loc[sub["y_true"] == 1, "y_prob"], bins=20, alpha=0.6, label="true PREICTAL", density=True)
    ax.axvline(THRESHOLD, color="black", linestyle=":", label=f"threshold={THRESHOLD}")
    ax.set_title(f"H = {H} min")
    ax.set_xlabel("Predicted P(PREICTAL)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Density")
fig.suptitle("Predicted PREICTAL probability by true label (TCN, pooled evaluated folds)")
fig.tight_layout()
fig.savefig(plot_dir / "probability_distributions.png", dpi=150)
plt.close(fig)

print("[OK] Saved 3 diagnostic plot files to", plot_dir)
for p in sorted(plot_dir.glob("*.png")):
    print(" -", p.name)


[OK] Saved 3 diagnostic plot files to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\tcn_loso_binary\plots
 - confusion_matrices_tcn.png
 - pr_curves_overall_by_horizon.png
 - probability_distributions.png


## 14. Save outputs

New, clearly-named directory: `outputs/modeling/tcn_loso_binary/`. Nothing
existing (including `outputs/modeling/logistic_loso_binary/` and
`outputs/modeling/random_forest_loso_binary/`) is overwritten.


In [15]:

predictions_df.to_parquet(run_dir / "predictions.parquet", index=False)
predictions_df.to_csv(run_dir / "predictions.csv", index=False)
fold_metrics_df.to_csv(run_dir / "fold_metrics.csv", index=False)
patient_horizon_summary_df.to_csv(run_dir / "patient_horizon_summary.csv", index=False)
overall_horizon_summary_df.to_csv(run_dir / "overall_horizon_summary.csv", index=False)
fold_diagnostics_df.to_csv(run_dir / "fold_sequence_diagnostics.csv", index=False)
skipped_df.to_csv(run_dir / "skipped_folds.csv", index=False)

experiment_config = {
    "task": "Binary PREICTAL vs INTERICTAL detection, patient-specific LOSO, TCN (temporal) baseline",
    "model": "TinyTCNBinary (causal, dilated Conv1d, 3 blocks)",
    "model_params": {"hidden": TCN_HIDDEN, "kernel_size": TCN_KERNEL_SIZE, "dilations": [1, 2, 4]},
    "sequence_length_epochs": SEQUENCE_LENGTH,
    "sequence_length_minutes": SEQUENCE_LENGTH * EPOCH_SEC / 60,
    "sequence_rule": "same recording, contiguous epoch_idx, ALL epochs in the window in the SAME fold split (never crosses TRAIN/TEST)",
    "training": {
        "optimizer": "Adam", "lr": TCN_LR, "batch_size_max": TCN_BATCH_SIZE,
        "max_epochs": TCN_MAX_EPOCHS, "fallback_epochs_no_val": TCN_FALLBACK_EPOCHS,
        "early_stopping_patience": TCN_PATIENCE, "early_stopping_min_delta": TCN_MIN_DELTA,
        "loss": "BCEWithLogitsLoss with pos_weight = n_train_neg / n_train_pos (from sub-TRAIN)",
        "validation_source": "TRAIN only (recording-level holdout, or chronological gap fallback), Section 7",
    },
    "naive_baseline": "DummyClassifier(strategy='most_frequent')",
    "random_state": RANDOM_STATE,
    "decision_threshold": THRESHOLD,
    "threshold_tuned_on_test": False,
    "hyperparameter_tuning_performed": False,
    "horizons_minutes": HORIZONS,
    "patients": PATIENTS,
    "patients_pooled_across": False,
    "feature_cols": FEATURE_COLS,
    "epoch_sec": EPOCH_SEC,
    "n_folds_attempted": int(n_folds_attempted),
    "n_folds_evaluated": int(n_folds_evaluated),
    "n_folds_skipped": int(len(skipped_df)),
    "source_labels_table": str(in_path),
    "reference_lr_run": str(lr_run_dir),
    "reference_rf_run": str(rf_run_dir),
    "normalization": "per-fold, per-patient StandardScaler fit on TRAIN rows only (reused verbatim from EEG_patient_normalization.ipynb / EEG_loso_evaluation_splitting.ipynb / EEG_logistic_loso_binary.ipynb)",
    "metrics_framework_source": "EEG_seizure_prediction_metrics.ipynb / EEG_logistic_loso_binary.ipynb",
}
with open(run_dir / "experiment_config.json", "w") as f:
    _json.dump(experiment_config, f, indent=2, default=str)

print("Saved to", run_dir, ":")
for p in sorted(run_dir.glob("*")):
    if p.is_file():
        print(" -", p.name)

for f in ["labels_binary_preictal.parquet", "loso_fold_manifest.parquet"]:
    assert (output_dir / f).exists()
for d, name in [(lr_run_dir, "LR"), (rf_run_dir, "RF")]:
    for f in ["predictions.parquet", "fold_metrics.csv", "overall_horizon_summary.csv", "experiment_config.json"]:
        assert (d / f).exists(), f"{name} output {f} missing"
print("\n[OK] All previously existing outputs, including LR's and RF's, remain untouched.")


Saved to C:\Users\angela\Downloads\EEG_timeseries-main\EEG_timeseries-main\outputs\modeling\tcn_loso_binary :
 - experiment_config.json
 - fold_metrics.csv
 - fold_sequence_diagnostics.csv
 - lr_vs_rf_vs_tcn_comparison_by_patient.csv
 - lr_vs_rf_vs_tcn_comparison_overall.csv
 - overall_horizon_summary.csv
 - patient_horizon_summary.csv
 - predictions.csv
 - predictions.parquet
 - skipped_folds.csv

[OK] All previously existing outputs, including LR's and RF's, remain untouched.


## 15. Three-way comparison: Logistic Regression vs Random Forest vs TCN

Reads LR's and RF's numbers directly from their own saved
`overall_horizon_summary.csv` / `patient_horizon_summary.csv` files (the
actual outputs from those already-validated, already-executed runs) --
**never hardcoded**. TCN's numbers are this run's own (already saved above,
Section 14) and are explicitly coverage-limited (Section 10) -- flagged
in every comparison row.


In [16]:

def _load_overall(run_dir, model_key):
    d = pd.read_csv(run_dir / "overall_horizon_summary.csv")
    return d[d["model"] == model_key].copy()

def _load_patient(run_dir, model_key):
    d = pd.read_csv(run_dir / "patient_horizon_summary.csv")
    return d[d["model"] == model_key].copy()

lr_overall = _load_overall(lr_run_dir, "logistic_regression")
rf_overall = _load_overall(rf_run_dir, "random_forest")
tcn_overall = overall_horizon_summary_df[overall_horizon_summary_df["model"] == "tcn"].copy()

for d, name in [(lr_overall, "LR"), (rf_overall, "RF")]:
    d["n_folds_evaluated"] = 19  # LR/RF: full epoch-level coverage, one row per eligible seizure, all 19/19
    d["coverage_note"] = "19/19 eligible seizures (epoch-level, no sequence-history requirement)"
tcn_overall["coverage_note"] = tcn_overall.apply(
    lambda r: f"{int(r['n_folds_evaluated'])}/{int(r['n_eligible_seizures'])} eligible seizures "
              f"(sequence-level: needs {SEQUENCE_LENGTH} contiguous same-split epochs)", axis=1)

combined_overall = pd.concat([lr_overall, rf_overall, tcn_overall], ignore_index=True, sort=False)
combined_overall["detected_over_eligible"] = (
    combined_overall["n_seizures_detected"].astype(int).astype(str) + "/" + combined_overall["n_folds"].astype(int).astype(str)
)

print("=" * 110)
print("LOGISTIC REGRESSION vs RANDOM FOREST vs TCN -- POOLED OVERALL, BY HORIZON")
print("=" * 110)
comp_cols = ["horizon_minutes", "model", "seizure_sensitivity", "detected_over_eligible",
             "false_alarms_per_hour", "auc_pr", "precision_preictal", "recall_preictal", "f1_preictal", "coverage_note"]
print(combined_overall.sort_values(["horizon_minutes", "model"])[comp_cols].to_string(index=False))

lr_patient = _load_patient(lr_run_dir, "logistic_regression")
rf_patient = _load_patient(rf_run_dir, "random_forest")
tcn_patient = patient_horizon_summary_df[patient_horizon_summary_df["model"] == "tcn"].copy()
for d in (lr_patient, rf_patient):
    d["n_folds_evaluated"] = d["n_folds"]
combined_patient = pd.concat([lr_patient, rf_patient, tcn_patient], ignore_index=True, sort=False)
combined_patient["detected_over_eligible"] = (
    combined_patient["n_seizures_detected"].astype(int).astype(str) + "/" + combined_patient["n_folds"].astype(int).astype(str)
)

print("\n" + "=" * 110)
print("LOGISTIC REGRESSION vs RANDOM FOREST vs TCN -- PER PATIENT + HORIZON")
print("=" * 110)
comp_patient_cols = ["patient_id", "horizon_minutes", "model", "seizure_sensitivity", "detected_over_eligible",
                      "false_alarms_per_hour", "auc_pr", "precision_preictal", "recall_preictal", "f1_preictal"]
print(combined_patient.sort_values(["patient_id", "horizon_minutes", "model"])[comp_patient_cols].to_string(index=False))

combined_overall.sort_values(["horizon_minutes", "model"])[comp_cols].to_csv(run_dir / "lr_vs_rf_vs_tcn_comparison_overall.csv", index=False)
combined_patient.sort_values(["patient_id", "horizon_minutes", "model"])[comp_patient_cols].to_csv(run_dir / "lr_vs_rf_vs_tcn_comparison_by_patient.csv", index=False)
print(f"\n[OK] Saved 3-way comparison tables to {run_dir}")

print("\n[LIMITATION, still present for ALL THREE models] CHB12 folds have 0 TEST interictal epochs at every "
      "horizon (project_state.md Section 20.5) -- false_alarms_per_hour and auc_pr are undefined (NaN) for "
      "CHB12 for all three models. All pooled FA/hour and AUC-PR numbers above are driven entirely by CHB24.")


LOGISTIC REGRESSION vs RANDOM FOREST vs TCN -- POOLED OVERALL, BY HORIZON
 horizon_minutes               model  seizure_sensitivity detected_over_eligible  false_alarms_per_hour   auc_pr  precision_preictal  recall_preictal  f1_preictal                                                                   coverage_note
               5 logistic_regression             0.947368                  18/19              13.652695 0.535829            0.348348         0.682353     0.461233          19/19 eligible seizures (epoch-level, no sequence-history requirement)
               5       random_forest             0.842105                  16/19              16.886228 0.528692            0.388000         0.570588     0.461905          19/19 eligible seizures (epoch-level, no sequence-history requirement)
               5                 tcn             0.937500                  15/16               3.157895 0.362504            0.053381         1.000000     0.101351 16/19 eligible seizures (sequence-

## 16. Scientific interpretation

In [17]:

def _get(df, H, model):
    row = df[(df["horizon_minutes"] == H) & (df["model"] == model)]
    return row.iloc[0] if len(row) else None

def _fmt(x, nd=3):
    return "NaN" if (x is None or pd.isna(x)) else f"{x:.{nd}f}"

lines = []

lines.append("### 1. Does RF outperform LR?")
for H in HORIZONS:
    l, r = _get(combined_overall, H, "logistic_regression"), _get(combined_overall, H, "random_forest")
    lines.append(f"  H={H:2d}min: LR AUC-PR={_fmt(l['auc_pr'])}, RF AUC-PR={_fmt(r['auc_pr'])}, "
                 f"LR FA/hr={_fmt(l['false_alarms_per_hour'])}, RF FA/hr={_fmt(r['false_alarms_per_hour'])}, "
                 f"LR sens={_fmt(l['seizure_sensitivity'])}, RF sens={_fmt(r['seizure_sensitivity'])}")
lines.append("  RF's AUC-PR gain over LR is modest and horizon-dependent (negative at H=5, positive at "
             "H=15/30); RF's FA/hour is consistently WORSE than LR's. Not a clear win either way "
             "(full detail: project_state.md Section 23.4).")

lines.append("\n### 2. Does TCN outperform LR?")
for H in HORIZONS:
    l, t = _get(combined_overall, H, "logistic_regression"), _get(combined_overall, H, "tcn")
    lines.append(f"  H={H:2d}min: LR AUC-PR={_fmt(l['auc_pr'])}, TCN AUC-PR={_fmt(t['auc_pr'] if t is not None else np.nan)} "
                 f"(TCN coverage: {t['coverage_note'] if t is not None else 'n/a'})")

lines.append("\n### 3. Does TCN outperform RF?")
for H in HORIZONS:
    r, t = _get(combined_overall, H, "random_forest"), _get(combined_overall, H, "tcn")
    lines.append(f"  H={H:2d}min: RF AUC-PR={_fmt(r['auc_pr'])}, TCN AUC-PR={_fmt(t['auc_pr'] if t is not None else np.nan)}")

lines.append("\n### 4. Which horizon appears strongest (across all 3 models)?")
for model_name in ["logistic_regression", "random_forest", "tcn"]:
    sub = combined_overall[combined_overall["model"] == model_name].sort_values("auc_pr", ascending=False)
    if len(sub):
        best = sub.iloc[0]
        lines.append(f"  {model_name}: best AUC-PR at H={int(best['horizon_minutes'])}min ({_fmt(best['auc_pr'])})")

lines.append("\n### 5. Which model gives the best AUC-PR (per horizon)?")
for H in HORIZONS:
    sub = combined_overall[combined_overall["horizon_minutes"] == H].dropna(subset=["auc_pr"])
    if len(sub):
        best = sub.loc[sub["auc_pr"].idxmax()]
        lines.append(f"  H={H:2d}min: best = {best['model']} (AUC-PR={_fmt(best['auc_pr'])})")

lines.append("\n### 6. Which gives the best sensitivity / false-alarm tradeoff?")
for H in HORIZONS:
    sub = combined_overall[(combined_overall["horizon_minutes"] == H) & (combined_overall["model"] != "dummy_most_frequent")]
    sub = sub.dropna(subset=["false_alarms_per_hour"])
    if len(sub):
        for _, row in sub.iterrows():
            lines.append(f"  H={H:2d}min {row['model']}: sensitivity={_fmt(row['seizure_sensitivity'])}, FA/hr={_fmt(row['false_alarms_per_hour'])}")

lines.append("\n### 7. Are results dominated by CHB24 because CHB12 lacks TEST interictal data?")
lines.append("  Yes, for all three models -- CHB12 contributes sensitivity only (FA/hour, AUC-PR are NaN "
             "for CHB12 at every horizon, every model, Section 15's limitation note). Every pooled FA/hour "
             "and AUC-PR figure above is mechanically CHB24-only.")

h30_train_prev_note = "all 19/19 LOSO folds have TRAIN PREICTAL prevalence > 0.5 at H=30 (project_state.md Section 22.2/23.4.7)"
lines.append("\n### 8. Is H=30 still vulnerable to the majority-class/dummy-baseline issue?")
d30 = _get(combined_overall, 30, "dummy_most_frequent")
lines.append(f"  Yes -- {h30_train_prev_note}. dummy_most_frequent at H=30: sensitivity={_fmt(d30['seizure_sensitivity']) if d30 is not None else 'n/a'}, "
             f"FA/hr={_fmt(d30['false_alarms_per_hour']) if d30 is not None else 'n/a'} -- a naive constant-positive-leaning "
             f"predictor already looks strong at H=30 for the same structural reason documented for LR and RF.")

lines.append("\n### 8b. [CRITICAL CAVEAT] Is the TCN's FA/hour figure trustworthy?")
pos_rates_tcn = pos_rate_df[pos_rate_df["model"] == "tcn"]
lines.append("  NO, not at face value. TCN's TEST predicted-positive rate (Section 12) is "
             + ", ".join(f"H={int(r['horizon_minutes'])}min={r['predicted_positive_rate']:.3f}" for _, r in pos_rates_tcn.iterrows())
             + " -- i.e. the TCN predicts PREICTAL on almost every TEST sequence at EVERY horizon "
             "(confirmed degenerate, >98% positive, at H=5 and H=30 by the same flag LR/RF are checked "
             "against). Confusion matrices (Section 12) show CHB24 TN=0 at every horizon -- 100% of CHB24's "
             "interictal TEST sequences are false positives. The apparently LOW pooled FA/hour (3.16/hour, "
             "far below LR's 12.6-13.7 and RF's 15.8-18.3) is an ARTIFACT of the alarm-event-merging "
             "convention (project_state.md Section 22.1): a near-constant positive prediction collapses into "
             "very few countable 'alarm events' per test period, not genuine specificity. This mirrors the "
             "documented H=30 dummy-baseline artifact (question 8) but is now present for the TCN's actual "
             "model output, at every horizon, not just H=30. TCN's headline AUC-PR numbers (threshold-"
             "independent, based on probability ranking) are less affected by this and may still carry some "
             "signal, but its FIXED-THRESHOLD (0.5) sensitivity/FA-hour/precision/accuracy numbers above "
             "should NOT be read as 'TCN has fewer false alarms than LR/RF' -- it is closer to an "
             "under-trained, heavily class-weighted model that has collapsed toward predicting the minority "
             "(PREICTAL) class almost everywhere. Likely cause: pos_weight for the most imbalanced folds "
             "(e.g. CHB24 H=5, ~266 negative vs ~10-11 positive sub-train sequences) is very large (~24-27x), "
             "early stopping fires quickly (mean 14.9 epochs trained, Section 12), and this compact "
             "architecture with a fixed 1e-3 learning rate and 0.5 threshold was not calibrated to handle "
             "that combination well. This is a genuine limitation of THIS baseline configuration, not fixed "
             "in this sprint (no hyperparameter/threshold tuning was in scope) -- flagged here explicitly "
             "rather than allowed to read as a real improvement.")

lines.append("\n### 9. Does temporal modeling provide measurable benefit?")
tcn_valid = combined_overall[(combined_overall["model"] == "tcn")].dropna(subset=["auc_pr"])
rf_valid = combined_overall[(combined_overall["model"] == "random_forest")]
if len(tcn_valid):
    deltas = []
    for H in HORIZONS:
        t, r = _get(combined_overall, H, "tcn"), _get(combined_overall, H, "random_forest")
        if t is not None and r is not None and not pd.isna(t["auc_pr"]):
            deltas.append((H, t["auc_pr"] - r["auc_pr"]))
    lines.append(f"  TCN vs RF AUC-PR delta by horizon: " + ", ".join(f"H={h}:{_fmt(d)}" for h, d in deltas))
    lines.append("  See numeric deltas above -- interpret alongside TCN's reduced fold coverage (Section 10) "
                 "before concluding temporal modeling helps or hurts.")
else:
    lines.append("  TCN produced no evaluable folds with AUC-PR defined -- cannot assess.")

lines.append("\n### 10. Is the current channel-averaged feature representation likely limiting all three models?")
lines.append("  All three models share the identical 14 qEEG features and identical folds; if AUC-PR deltas "
             "between the linear (LR), tree-ensemble (RF), and temporal (TCN) model families are all modest, "
             "that is more consistent with the feature representation being the shared bottleneck than with "
             "any one estimator family being clearly superior -- consistent with project_state.md's roadmap "
             "toward raw-EEG/channel-wise representations (Section 9) as the next lever, not further "
             "classical-ML or architecture search on the current 14-feature representation.")

lines.append("\n### 11. Are there signs of overfitting?")
if len(overfit_df):
    lines.append("  TCN mean TRAIN accuracy vs pooled TEST accuracy, per patient/horizon:")
    for _, r in overfit_df.iterrows():
        lines.append(f"    {r['patient_id']} H={int(r['horizon_minutes']):2d}min: TRAIN acc={_fmt(r['mean_train_accuracy'])}, "
                     f"TEST acc={_fmt(r['pooled_test_accuracy'])}, gap={_fmt(r['gap'])}")
lines.append("  RF's overfitting (CHB24 TRAIN acc ~0.996-1.0 vs TEST acc ~0.46-0.56, project_state.md "
             "Section 23.4.6) was already confirmed as a real limitation of that baseline; the TCN gap above "
             "is the same style of diagnostic for the temporal model.")

lines.append("\n### 12. What conclusions are scientifically defensible with only two patients?")
lines.append(f"  Very limited ones. This experiment spans 2 patients, {int(overall_horizon_summary_df['n_eligible_seizures'].iloc[0]) if len(overall_horizon_summary_df) else 19} "
             f"total eligible seizures per horizon (LR/RF), and TCN evaluates a strict subset of those "
             f"({n_folds_evaluated} of 57 fold-rows total, Section 10) due to its own sequence-history "
             f"requirement. No statistical significance testing was performed on any pairwise model "
             f"difference. None of the LR/RF/TCN differences reported above should be read as a "
             f"generalizable claim about which model family is 'better' for seizure forecasting -- only as "
             f"descriptive results on this specific 2-patient, 14-feature, LOSO-evaluated sample.")

print("\n".join(lines))


### 1. Does RF outperform LR?
  H= 5min: LR AUC-PR=0.536, RF AUC-PR=0.529, LR FA/hr=13.653, RF FA/hr=16.886, LR sens=0.947, RF sens=0.842
  H=15min: LR AUC-PR=0.650, RF AUC-PR=0.731, LR FA/hr=13.293, RF FA/hr=18.323, LR sens=0.947, RF sens=0.947
  H=30min: LR AUC-PR=0.736, RF AUC-PR=0.768, LR FA/hr=12.575, RF FA/hr=15.808, LR sens=0.947, RF sens=0.947
  RF's AUC-PR gain over LR is modest and horizon-dependent (negative at H=5, positive at H=15/30); RF's FA/hour is consistently WORSE than LR's. Not a clear win either way (full detail: project_state.md Section 23.4).

### 2. Does TCN outperform LR?
  H= 5min: LR AUC-PR=0.536, TCN AUC-PR=0.363 (TCN coverage: 16/19 eligible seizures (sequence-level: needs 10 contiguous same-split epochs))
  H=15min: LR AUC-PR=0.650, TCN AUC-PR=0.550 (TCN coverage: 16/19 eligible seizures (sequence-level: needs 10 contiguous same-split epochs))
  H=30min: LR AUC-PR=0.736, TCN AUC-PR=0.794 (TCN coverage: 16/19 eligible seizures (sequence-level: needs 10 cont